# Experiment 1 Probe Training from Zip Archive

Use this after `colab_exp1_zip_feature_extraction.ipynb` has copied feature caches to Drive. This notebook unzips the compressed project into local Colab disk, restores feature caches from Drive, trains global and dense probes, then copies outputs back to Drive.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')
ZIP_PATH = Path('/content/drive/MyDrive/cv-project-exp1-rerender-colab.zip')
UNPACK_BASE = Path('/content/cv-project')
DRIVE_RESULTS = Path('/content/drive/MyDrive/cv-project-exp1-rerender-results')


def _find_exp1_root(base: Path) -> Path:
    """Zip archives often add one top-level folder; resolve to the tree that has exp1/."""
    marker = base / 'exp1' / 'features' / '__init__.py'
    if marker.is_file():
        return base
    for child in sorted(base.iterdir()):
        if child.is_dir() and (child / 'exp1' / 'features' / '__init__.py').is_file():
            return child
    raise FileNotFoundError(
        f'Could not find exp1 package under {base}. '
        'Re-zip the repo so it includes the full exp1/ directory.'
    )


assert ZIP_PATH.is_file(), f'Missing zip file: {ZIP_PATH}'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

!rm -rf /content/cv-project
!mkdir -p /content/cv-project
!unzip -q -o "{ZIP_PATH}" -d "{UNPACK_BASE}"

WORKDIR = _find_exp1_root(UNPACK_BASE)
os.environ['CV_PROJECT_ROOT'] = str(WORKDIR)
os.environ['PYTHONPATH'] = str(WORKDIR)
%cd {WORKDIR}
!nvidia-smi

In [ ]:
import sys

!{sys.executable} -m pip install -q -r requirements.txt
!{sys.executable} -m pip install -q pyarrow matplotlib seaborn scikit-learn

In [ ]:
# Rewrite manifest path columns to relative paths so the dense probe scripts
# can resolve them against the Colab project root. Older zips contain absolute
# Mac paths (`/Users/jerry/cv-project/...`) which `_resolve_path` would otherwise
# pass through unchanged, causing FileNotFoundError on Colab. Idempotent.
import pandas as pd
from pathlib import Path

PATH_COLUMNS = (
    'raw_mesh_path',
    'normalized_mesh_path',
    'rgb_path',
    'depth_path',
    'normal_path',
    'mask_path',
    'mesh_import_path',
    'random_texture_path',
)
MANIFESTS = [
    'data/exp1_under12h/manifests/render_valid.parquet',
    'data/exp1_under12h/manifests/render_qc.parquet',
    'data/exp1_under12h_dense/manifests/render_valid.parquet',
    'data/exp1_under12h_dense/manifests/render_qc.parquet',
]
ANCHORS = ('data', 'configs', 'exp1', 'scripts', 'src')


def _relativize(value):
    if value is None:
        return value
    text = str(value)
    if not text or text.lower() == 'nan':
        return text
    path = Path(text)
    if not path.is_absolute():
        return text
    parts = path.parts
    for anchor in ANCHORS:
        if anchor in parts:
            return str(Path(*parts[parts.index(anchor):]))
    return text


for rel in MANIFESTS:
    p = WORKDIR / rel
    if not p.is_file():
        print(f'skipped (missing): {rel}')
        continue
    df = pd.read_parquet(p)
    changed_cols = []
    for col in PATH_COLUMNS:
        if col not in df.columns:
            continue
        before = df[col].astype('object')
        after = before.map(_relativize)
        if not (before == after).all():
            df[col] = after
            changed_cols.append(col)
    if changed_cols:
        df.to_parquet(p, index=False)
        print(f'rewrote {rel}: {changed_cols}')
    else:
        print(f'already relative: {rel}')

In [ ]:
# Restore feature caches generated by the zip feature-extraction notebook.
import shutil

for rel in ['data/exp1_under12h/features', 'data/exp1_under12h_dense/features']:
    src = DRIVE_RESULTS / rel
    dst = WORKDIR / rel
    if not src.exists():
        raise FileNotFoundError(f'Missing feature cache directory on Drive: {src}')
    if dst.exists():
        shutil.rmtree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst)
    print(f'Restored {src} -> {dst}')

In [ ]:
# Global frozen-feature linear probes.
!PYTHONPATH=. python scripts/train_all_exp1_probes.py --config configs/exp1_under12h.yaml --device cuda --batch-size 256

In [ ]:
# Dense patch-depth probes over the dense subset.
!PYTHONPATH=. python scripts/train_all_dense_depth_probes.py --config configs/exp1_under12h_dense.yaml --device cuda --batch-size 256

In [ ]:
# Dense patch surface-normal probes over the same dense subset.
!PYTHONPATH=. python scripts/train_all_dense_surface_normal_probes.py --config configs/exp1_under12h_dense.yaml --device cuda --batch-size 256

In [ ]:
# Aggregate results and regenerate figures.
!PYTHONPATH=. python scripts/aggregate_exp1_results.py --config configs/exp1_under12h.yaml
!PYTHONPATH=. python scripts/make_exp1_figures.py --config configs/exp1_under12h.yaml
!PYTHONPATH=. python scripts/aggregate_exp1_results.py --config configs/exp1_under12h_dense.yaml
!PYTHONPATH=. python scripts/make_exp1_figures.py --config configs/exp1_under12h_dense.yaml

In [ ]:
# Copy probe outputs, result tables, and figures back to Drive.
import shutil

for rel in ['outputs/exp1_under12h', 'outputs/exp1_under12h_dense']:
    src = WORKDIR / rel
    dst = DRIVE_RESULTS / rel
    if dst.exists():
        shutil.rmtree(dst)
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst)
        print(f'Copied {src} -> {dst}')
    else:
        print(f'Skipped missing {src}')